# **R : Experiment - 6**

Mohd Talha Patrawala

CMPN - B

23102B0025

In [1]:
url <- "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv"
penguins <- read.csv(url, stringsAsFactors = TRUE)

penguins <- penguins[complete.cases(penguins[, c("body_mass_g", "flipper_length_mm",
                                                  "species", "sex")]), ]
cat("Number of penguins after removing missing values:", nrow(penguins), "\n")
str(penguins)

skewness <- function(x) {
  x <- x[!is.na(x)]
  n <- length(x)
  m <- mean(x)
  s <- sd(x)
  (sum((x - m)^3) / n) / s^3
}

excess_kurtosis <- function(x) {
  x <- x[!is.na(x)]
  n <- length(x)
  m <- mean(x)
  s <- sd(x)
  (sum((x - m)^4) / n) / s^4 - 3
}

levene_test <- function(y, group) {
  group <- factor(group)
  centered <- ave(y, group, FUN = function(v) abs(v - median(v)))
  fit <- aov(centered ~ group)
  summary(fit)[[1]]
}

cohend <- function(x, y) {
  nx <- length(x); ny <- length(y)
  sp <- sqrt(((nx - 1) * var(x) + (ny - 1) * var(y)) / (nx + ny - 2))
  (mean(x) - mean(y)) / sp
}

descriptives <- function(x) {
  x <- x[!is.na(x)]
  c(Mean = mean(x), Median = median(x), Min = min(x), Max = max(x),
    Variance = var(x), SD = sd(x),
    Q1 = as.numeric(quantile(x, 0.25)), Q3 = as.numeric(quantile(x, 0.75)),
    IQR = IQR(x), Skewness = skewness(x), Kurtosis = excess_kurtosis(x))
}


Number of penguins after removing missing values: 333 
'data.frame':	333 obs. of  8 variables:
 $ species          : Factor w/ 3 levels "Adelie","Chinstrap",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ island           : Factor w/ 3 levels "Biscoe","Dream",..: 3 3 3 3 3 3 3 3 3 3 ...
 $ bill_length_mm   : num  39.1 39.5 40.3 36.7 39.3 38.9 39.2 41.1 38.6 34.6 ...
 $ bill_depth_mm    : num  18.7 17.4 18 19.3 20.6 17.8 19.6 17.6 21.2 21.1 ...
 $ flipper_length_mm: int  181 186 195 193 190 181 195 182 191 198 ...
 $ body_mass_g      : int  3750 3800 3250 3450 3650 3625 4675 3200 3800 4400 ...
 $ sex              : Factor w/ 2 levels "female","male": 2 1 1 1 2 1 2 1 2 2 ...
 $ year             : int  2007 2007 2007 2007 2007 2007 2007 2007 2007 2007 ...


In [2]:
cat("\n===== 1. Descriptive statistics: body_mass_g (overall) =====\n")
overall_desc <- descriptives(penguins$body_mass_g)
print(round(overall_desc, 2))

cat("\n===== Descriptive statistics: body_mass_g by species =====\n")
species_desc <- sapply(split(penguins$body_mass_g, penguins$species), descriptives)
print(round(species_desc, 2))

png("hist_body_mass.png", width = 800, height = 600)
hist(penguins$body_mass_g, breaks = 20, col = "steelblue",
     main = "Histogram of Penguin Body Mass", xlab = "Body Mass (g)", ylab = "Frequency")
dev.off()

png("boxplot_body_mass_species.png", width = 800, height = 600)
boxplot(body_mass_g ~ species, data = penguins, col = c("skyblue", "salmon", "palegreen"),
        main = "Body Mass by Species", xlab = "Species", ylab = "Body Mass (g)")
dev.off()

png("density_body_mass.png", width = 800, height = 600)
d <- density(penguins$body_mass_g)
plot(d, main = "Density Plot of Body Mass", xlab = "Body Mass (g)", col = "darkred", lwd = 2)
polygon(d, col = rgb(1, 0, 0, 0.2), border = "darkred")
dev.off()


===== 1. Descriptive statistics: body_mass_g (overall) =====
     Mean    Median       Min       Max  Variance        SD        Q1        Q3 
  4207.06   4050.00   2700.00   6300.00 648372.49    805.22   3550.00   4775.00 
      IQR  Skewness  Kurtosis 
  1225.00      0.47     -0.75 

===== Descriptive statistics: body_mass_g by species =====
            Adelie Chinstrap    Gentoo
Mean       3706.16   3733.09   5092.44
Median     3700.00   3700.00   5050.00
Min        2850.00   2700.00   3950.00
Max        4775.00   4800.00   6300.00
Variance 210332.43 147713.45 251478.33
SD          458.62    384.34    501.48
Q1         3362.50   3487.50   4700.00
Q3         4000.00   3950.00   5500.00
IQR         637.50    462.50    800.00
Skewness      0.27      0.24      0.04
Kurtosis     -0.62      0.36     -0.79


agg_record_117662162b16 
                      2

agg_record_117662162b16 
                      2

agg_record_117662162b16 
                      2

In [3]:
cat("\n===== 2. Hypothesis testing: body mass by sex =====\n")
# H0: mu_male = mu_female  (no difference in mean body mass between sexes)
# H1: mu_male != mu_female (mean body mass differs by sex)

male_mass   <- penguins$body_mass_g[penguins$sex == "male"]
female_mass <- penguins$body_mass_g[penguins$sex == "female"]

cat("\n-- Shapiro-Wilk normality test --\n")
print(shapiro.test(male_mass))
print(shapiro.test(female_mass))

png("qq_male_female.png", width = 900, height = 500)
par(mfrow = c(1, 2))
qqnorm(male_mass, main = "QQ-Plot: Male Body Mass"); qqline(male_mass, col = "red")
qqnorm(female_mass, main = "QQ-Plot: Female Body Mass"); qqline(female_mass, col = "red")
dev.off()
par(mfrow = c(1, 1))

cat("\n-- Levene's test for equality of variances (sex) --\n")
print(levene_test(penguins$body_mass_g, penguins$sex))

cat("\n-- Welch two-sample t-test --\n")
ttest_result <- t.test(male_mass, female_mass, var.equal = FALSE)
print(ttest_result)

cat("\n-- Cohen's d (effect size) --\n")
d_value <- cohend(male_mass, female_mass)
cat("Cohen's d =", round(d_value, 3), "\n")

cat("\nInterpretation:\n")
cat("The p-value is far below 0.05, so we reject H0: male and female\n")
cat("penguins differ significantly in mean body mass. The 95% CI for the\n")
cat("difference (male - female) does not include 0, and Cohen's d indicates\n")
cat("a large effect size (|d| close to or above 0.8).\n")


===== 2. Hypothesis testing: body mass by sex =====

-- Shapiro-Wilk normality test --

	Shapiro-Wilk normality test

data:  male_mass
W = 0.92504, p-value = 1.227e-07


	Shapiro-Wilk normality test

data:  female_mass
W = 0.91931, p-value = 6.155e-08



agg_record_11763356634c 
                      2


-- Levene's test for equality of variances (sex) --
             Df   Sum Sq Mean Sq F value  Pr(>F)  
group         1  1245475 1245475  6.0586 0.01435 *
Residuals   331 68043921  205571                  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

-- Welch two-sample t-test --

	Welch Two Sample t-test

data:  male_mass and female_mass
t = 8.5545, df = 323.9, p-value = 4.794e-16
alternative hypothesis: true difference in means is not equal to 0
95 percent confidence interval:
 526.2453 840.5783
sample estimates:
mean of x mean of y 
 4545.685  3862.273 


-- Cohen's d (effect size) --
Cohen's d = 0.936 

Interpretation:
The p-value is far below 0.05, so we reject H0: male and female
penguins differ significantly in mean body mass. The 95% CI for the
difference (male - female) does not include 0, and Cohen's d indicates
a large effect size (|d| close to or above 0.8).


In [4]:
cat("\n===== 3. One-way ANOVA: body_mass_g ~ species =====\n")

cat("\n-- Shapiro-Wilk normality per species --\n")
for (sp in levels(penguins$species)) {
  cat(sp, ":\n")
  print(shapiro.test(penguins$body_mass_g[penguins$species == sp]))
}

png("qq_species_body_mass.png", width = 1200, height = 450)
par(mfrow = c(1, 3))
for (sp in levels(penguins$species)) {
  x <- penguins$body_mass_g[penguins$species == sp]
  qqnorm(x, main = paste("QQ-Plot:", sp)); qqline(x, col = "red")
}
dev.off()
par(mfrow = c(1, 1))

cat("\n-- Levene's test for homogeneity of variance (species) --\n")
print(levene_test(penguins$body_mass_g, penguins$species))

cat("\n-- One-way ANOVA --\n")
aov_species <- aov(body_mass_g ~ species, data = penguins)
print(summary(aov_species))

cat("\n-- Tukey HSD post-hoc test --\n")
tukey_species <- TukeyHSD(aov_species)
print(tukey_species)

png("tukey_species_bodymass.png", width = 800, height = 600)
plot(tukey_species, las = 1)
dev.off()


===== 3. One-way ANOVA: body_mass_g ~ species =====

-- Shapiro-Wilk normality per species --
Adelie :

	Shapiro-Wilk normality test

data:  penguins$body_mass_g[penguins$species == sp]
W = 0.98116, p-value = 0.04232

Chinstrap :

	Shapiro-Wilk normality test

data:  penguins$body_mass_g[penguins$species == sp]
W = 0.98449, p-value = 0.5605

Gentoo :

	Shapiro-Wilk normality test

data:  penguins$body_mass_g[penguins$species == sp]
W = 0.98606, p-value = 0.2605



agg_record_11765af9b728 
                      2


-- Levene's test for homogeneity of variance (species) --
             Df   Sum Sq Mean Sq F value   Pr(>F)   
group         2   716555  358277  5.1349 0.006367 **
Residuals   330 23025093   69773                    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

-- One-way ANOVA --
             Df    Sum Sq  Mean Sq F value Pr(>F)    
species       2 145190219 72595110   341.9 <2e-16 ***
Residuals   330  70069447   212332                   
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

-- Tukey HSD post-hoc test --
  Tukey multiple comparisons of means
    95% family-wise confidence level

Fit: aov(formula = body_mass_g ~ species, data = penguins)

$species
                       diff       lwr       upr    p adj
Chinstrap-Adelie   26.92385 -132.3528  186.2005 0.916431
Gentoo-Adelie    1386.27259 1252.2897 1520.2554 0.000000
Gentoo-Chinstrap 1359.34874 1194.4304 1524.2671 0.000000



agg_record_11765af9b728 
                      2

In [5]:
cat("\n===== 4. Kruskal-Wallis test: body_mass_g ~ species =====\n")
kw_species <- kruskal.test(body_mass_g ~ species, data = penguins)
print(kw_species)

cat("\nComparison of ANOVA vs Kruskal-Wallis:\n")
cat("Both the one-way ANOVA and the Kruskal-Wallis test produce extremely\n")
cat("small p-values (p < 0.001), leading to the same conclusion: penguin\n")
cat("body mass differs significantly across species. The agreement between\n")
cat("the parametric and non-parametric tests indicates the ANOVA conclusion\n")
cat("is robust even though Levene's test flagged some heterogeneity of variance.\n")


===== 4. Kruskal-Wallis test: body_mass_g ~ species =====

	Kruskal-Wallis rank sum test

data:  body_mass_g by species
Kruskal-Wallis chi-squared = 212.09, df = 2, p-value < 2.2e-16


Comparison of ANOVA vs Kruskal-Wallis:
Both the one-way ANOVA and the Kruskal-Wallis test produce extremely
small p-values (p < 0.001), leading to the same conclusion: penguin
body mass differs significantly across species. The agreement between
the parametric and non-parametric tests indicates the ANOVA conclusion
is robust even though Levene's test flagged some heterogeneity of variance.


In [6]:
cat("\n===== 5. Two-way ANOVA: body_mass_g ~ species * sex =====\n")
aov_twoway <- aov(body_mass_g ~ species * sex, data = penguins)
print(summary(aov_twoway))

png("interaction_species_sex.png", width = 800, height = 600)
interaction.plot(penguins$species, penguins$sex, penguins$body_mass_g,
                  col = c("blue", "red"), lwd = 2, type = "b", pch = 16,
                  xlab = "Species", ylab = "Mean Body Mass (g)",
                  trace.label = "Sex", main = "Species x Sex Interaction on Body Mass")
dev.off()

cat("\nInterpretation: species, sex, and the species:sex interaction are all\n")
cat("statistically significant (p < 0.05). Body mass increases from Adelie/\n")
cat("Chinstrap to Gentoo, males are consistently heavier than females within\n")
cat("each species, and the size of the male-female gap itself differs by\n")
cat("species (largest in Gentoo), which is what the significant interaction reflects.\n")


===== 5. Two-way ANOVA: body_mass_g ~ species * sex =====
             Df    Sum Sq  Mean Sq F value   Pr(>F)    
species       2 145190219 72595110 758.358  < 2e-16 ***
sex           1  37090262 37090262 387.460  < 2e-16 ***
species:sex   2   1676557   838278   8.757 0.000197 ***
Residuals   327  31302628    95727                     
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


agg_record_1176571a155e 
                      2


Interpretation: species, sex, and the species:sex interaction are all
statistically significant (p < 0.05). Body mass increases from Adelie/
Chinstrap to Gentoo, males are consistently heavier than females within
each species, and the size of the male-female gap itself differs by
species (largest in Gentoo), which is what the significant interaction reflects.


In [7]:
cat("\n===== 6. Flipper length analysis by species =====\n")
flipper_desc <- sapply(split(penguins$flipper_length_mm, penguins$species), descriptives)
print(round(flipper_desc, 2))

cat("\n-- Shapiro-Wilk normality per species (flipper length) --\n")
for (sp in levels(penguins$species)) {
  cat(sp, ":\n")
  print(shapiro.test(penguins$flipper_length_mm[penguins$species == sp]))
}

cat("\n-- Levene's test (flipper length, species) --\n")
print(levene_test(penguins$flipper_length_mm, penguins$species))

cat("\n-- One-way ANOVA: flipper_length_mm ~ species --\n")
aov_flipper <- aov(flipper_length_mm ~ species, data = penguins)
print(summary(aov_flipper))

cat("\n-- Tukey HSD (flipper length) --\n")
tukey_flipper <- TukeyHSD(aov_flipper)
print(tukey_flipper)

cat("\n-- Kruskal-Wallis (flipper length) --\n")
kw_flipper <- kruskal.test(flipper_length_mm ~ species, data = penguins)
print(kw_flipper)

png("boxplot_flipper_species.png", width = 800, height = 600)
boxplot(flipper_length_mm ~ species, data = penguins, col = c("skyblue", "salmon", "palegreen"),
        main = "Flipper Length by Species", xlab = "Species", ylab = "Flipper Length (mm)")
dev.off()

cat("\nInterpretation: Flipper length differs significantly across species\n")
cat("(ANOVA and Kruskal-Wallis both p < 0.001), and all pairwise Tukey\n")
cat("comparisons are significant, unlike body mass where Adelie vs Chinstrap\n")
cat("was NOT significant. This shows flipper length separates all three\n")
cat("species more cleanly than body mass does (Gentoo has by far the\n")
cat("longest flippers, consistent with it also having the highest body mass).\n")



===== 6. Flipper length analysis by species =====
         Adelie Chinstrap Gentoo
Mean     190.10    195.82 217.24
Median   190.00    196.00 216.00
Min      172.00    178.00 203.00
Max      210.00    212.00 231.00
Variance  42.53     50.86  43.37
SD         6.52      7.13   6.59
Q1       186.00    191.00 212.00
Q3       195.00    201.00 221.50
IQR        9.00     10.00   9.50
Skewness   0.08     -0.01   0.36
Kurtosis   0.28     -0.13  -0.72

-- Shapiro-Wilk normality per species (flipper length) --
Adelie :

	Shapiro-Wilk normality test

data:  penguins$flipper_length_mm[penguins$species == sp]
W = 0.9934, p-value = 0.7427

Chinstrap :

	Shapiro-Wilk normality test

data:  penguins$flipper_length_mm[penguins$species == sp]
W = 0.98891, p-value = 0.8106

Gentoo :

	Shapiro-Wilk normality test

data:  penguins$flipper_length_mm[penguins$species == sp]
W = 0.96148, p-value = 0.00176


-- Levene's test (flipper length, species) --
             Df Sum Sq Mean Sq F value Pr(>F)
group      

agg_record_1176143c48de 
                      2


Interpretation: Flipper length differs significantly across species
(ANOVA and Kruskal-Wallis both p < 0.001), and all pairwise Tukey
comparisons are significant, unlike body mass where Adelie vs Chinstrap
was NOT significant. This shows flipper length separates all three
species more cleanly than body mass does (Gentoo has by far the
longest flippers, consistent with it also having the highest body mass).


In [8]:
png("boxplot_body_mass_sex.png", width = 800, height = 600)
boxplot(body_mass_g ~ sex, data = penguins, col = c("lightpink", "lightblue"),
        main = "Body Mass by Sex", xlab = "Sex", ylab = "Body Mass (g)")
dev.off()

png("group_comparison_species_sex.png", width = 900, height = 600)
boxplot(body_mass_g ~ species * sex, data = penguins,
        col = rep(c("lightpink", "lightblue"), 3), las = 2,
        main = "Body Mass by Species and Sex", ylab = "Body Mass (g)")
dev.off()

agg_record_11767d891352 
                      2

agg_record_11767d891352 
                      2